## 🎯 Learning Objectives
* Understand the challenges of debugging and tracing LLM-powered applications.
* Learn how to integrate and enable LangSmith for LangChain applications.
* Interpret LangSmith traces to debug agent reasoning, identify bottlenecks, and monitor performance.
* Recognize the value of LangSmith for development, evaluation, and production monitoring of LangChain agents.


## Debugging and Tracing with LangSmith

Building robust and reliable LLM-powered applications, especially those involving complex chains and agents, presents unique debugging challenges. Unlike traditional deterministic software, LLM applications are inherently non-deterministic. An agent's reasoning path can vary based on subtle changes in prompts, model weights, or even the order of tool outputs. This makes traditional breakpoint debugging insufficient.

Imagine trying to diagnose an issue in a complex manufacturing plant where robots make decisions autonomously, and their internal thought process is a black box. You'd need a sophisticated monitoring system to record every action, every decision, and every input/output at each stage. This is precisely the role LangSmith plays for your LangChain applications.

**LangSmith** is LangChain's proprietary platform designed for debugging, testing, evaluating, and monitoring LLM applications. It acts as a "flight recorder" for your LangChain runs, capturing every step, every LLM call, every tool invocation, and every intermediate thought process. This comprehensive tracing capability is indispensable for:

1.  **Debugging Non-Deterministic Behavior**: Understand *why* an agent made a particular decision or failed to achieve its goal by inspecting its internal monologue and tool interactions.
2.  **Performance Optimization**: Identify bottlenecks, slow LLM calls, or inefficient tool usage.
3.  **Cost Management**: Monitor token usage and API calls to keep costs in check.
4.  **Evaluation and A/B Testing**: Compare different prompt strategies, model versions, or agent configurations side-by-side.
5.  **Production Monitoring**: Gain insights into how your application performs in the wild, catching regressions or unexpected behaviors.

In 2026, LangSmith has become the de-facto standard for observability in the LangChain ecosystem, offering deep integration and powerful analytics that are difficult to replicate with generic logging solutions. It provides a visual interface to explore traces, making complex agentic workflows transparent and manageable.

### How LangSmith Works (Simplified):

When LangSmith is enabled, your LangChain components (LLMs, tools, chains, agents) are instrumented to automatically send data about their execution to the LangSmith platform. This data includes:

*   **Inputs and Outputs**: What went into and came out of each component.
*   **Intermediate Steps**: The internal thoughts of an agent, tool calls, and their results.
*   **Latency**: How long each step took.
*   **Token Usage**: The number of input and output tokens for LLM calls.
*   **Errors**: Any exceptions or failures that occurred.

This information is then aggregated and presented in a user-friendly UI, allowing you to drill down into individual runs and understand the flow of execution.


In [ ]:
# Ensure you have the necessary libraries installed
# pip install langchain langchain-openai langchain-community langsmith

import os
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_react_agent
from langchain import hub
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.prompts import PromptTemplate

# --- 1. Set up LangSmith Environment Variables ---
# For LangSmith to work, you need to set these environment variables.
# Replace 'YOUR_LANGSMITH_API_KEY' with your actual LangSmith API key.
# You can find this in your LangSmith account settings.
# It's recommended to load these from a .env file in a real application.

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = "YOUR_LANGSMITH_API_KEY" # Replace with your actual key
os.environ["LANGCHAIN_PROJECT"] = "LLM02-L11-LangSmith-Demo" # Name your project

# --- 2. Set up OpenAI API Key ---
# Replace 'YOUR_OPENAI_API_KEY' with your actual OpenAI API key.
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY" # Replace with your actual key

# --- 3. Define Tools for the Agent ---
# We'll use a simple search tool for this example.
tools = [
    DuckDuckGoSearchRun(name="DuckDuckGoSearch")
]

# --- 4. Initialize the LLM ---
# Using a modern OpenAI model. Ensure it's capable of function calling.
llm = ChatOpenAI(model="gpt-4o-2024-05-13", temperature=0)

# --- 5. Create an Agent Prompt ---
# We'll use a pre-built prompt from LangChain Hub for a ReAct agent.
prompt = hub.pull("hwchase17/react")

# --- 6. Create the Agent ---
# The create_react_agent function helps set up a ReAct agent with the given LLM, tools, and prompt.
agent = create_react_agent(llm, tools, prompt)

# --- 7. Create the Agent Executor ---
# The AgentExecutor is responsible for running the agent.
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True, # Set to True to see console output, but LangSmith captures everything anyway
    handle_parsing_errors=True # Good practice for robust agents
)

# --- 8. Run the Agent with LangSmith Tracing Enabled ---
# When the environment variables are set, every call to agent_executor.invoke()
# will automatically be traced and sent to LangSmith.

print("\n--- Running Agent Query 1 ---")
result1 = agent_executor.invoke({"input": "What is the capital of France and what is its current population?"})
print(f"Agent Output 1: {result1['output']}")

print("\n--- Running Agent Query 2 ---")
result2 = agent_executor.invoke({"input": "Who won the FIFA World Cup in 2022?"})
print(f"Agent Output 2: {result2['output']}")

print("\n--- LangSmith Traces Available ---")
print("Visit your LangSmith project dashboard to view these traces:")
print(f"https://smith.langchain.com/o/YOUR_ORGANIZATION_ID/p/{os.environ['LANGCHAIN_PROJECT']}/traces")
print("Remember to replace 'YOUR_ORGANIZATION_ID' with your actual LangSmith organization ID.")

# Example of a custom chain without an agent, also traced by LangSmith
print("\n--- Running a simple chain with LangSmith Tracing ---")
custom_prompt = PromptTemplate.from_template("Tell me a fun fact about {topic}.")
chain = custom_prompt | llm

result_chain = chain.invoke({"topic": "penguins"})
print(f"Chain Output: {result_chain.content}")

print("\n--- LangSmith Traces for Chain Also Available ---")
print("Check your LangSmith dashboard for the 'chain' run as well.")


### Interpreting LangSmith Traces and Use Cases

After running the code above, navigate to your LangSmith project dashboard (using the URL provided in the code output, remembering to replace `YOUR_ORGANIZATION_ID`). You will see a list of "runs" corresponding to each `agent_executor.invoke()` and `chain.invoke()` call.

#### What to Look For in the LangSmith UI:

1.  **Run List**: Each entry represents a complete execution of your agent or chain. You'll see the input, output, status (success/failure), duration, and token usage at a glance.
2.  **Detailed Trace View**: Click on any run to see its full trace. This is where the magic happens:
    *   **Hierarchical View**: The trace is presented as a tree structure, showing the nested calls. You'll see the main `AgentExecutor` run, which contains `Agent` runs, which in turn contain `LLM` calls (for reasoning) and `Tool` calls (e.g., `DuckDuckGoSearch`).
    *   **Inputs/Outputs**: For each step, you can inspect the exact inputs it received and the outputs it produced. This is crucial for understanding *why* an LLM generated a certain response or *what* data a tool returned.
    *   **Intermediate Steps**: For agents, you'll see the `Thought`, `Action`, `Action Input`, and `Observation` steps. This is the agent's internal monologue, revealing its reasoning process.
    *   **Latency**: Each step shows its duration, helping you pinpoint performance bottlenecks.
    *   **Token Usage**: For LLM calls, you'll see the input and output token counts, essential for cost analysis.
    *   **Errors**: If an error occurred, LangSmith highlights the failing step and provides the traceback, making debugging much faster.

#### Performance Trade-offs:

Enabling LangSmith tracing introduces a minimal overhead due to the network calls required to send trace data to the LangSmith platform. For most development and even many production scenarios, this overhead is negligible compared to the latency of LLM calls themselves. The immense value gained in terms of observability, debugging speed, and evaluation capabilities far outweighs this minor performance impact.

#### Typical Use Cases:

*   **Debugging Agent Hallucinations**: If your agent provides incorrect information, trace its steps to see if it misinterpreted the prompt, used the wrong tool, or processed tool output incorrectly.
*   **Optimizing Prompts**: Run multiple variations of a prompt and compare their traces in LangSmith to see which leads to more efficient or accurate agent behavior.
*   **Identifying Tool Failures**: If an agent is stuck or failing, LangSmith will show you exactly which tool call failed and why.
*   **Understanding Complex Chains**: For multi-step chains, LangSmith provides a clear visual flow, helping you understand how data transforms at each stage.
*   **Regression Testing**: After making changes, run a suite of test cases and compare the new traces against baseline traces to ensure no regressions were introduced.
*   **A/B Testing in Production**: Deploy different agent versions and use LangSmith to compare their real-world performance and user interactions.

By making the opaque world of LLM reasoning transparent, LangSmith transforms the development and maintenance of agentic AI applications from a guessing game into a data-driven, systematic process.


### Resources

*   **LangSmith Documentation**: The official and most comprehensive guide to LangSmith's features and usage.
    *   [https://docs.smith.langchain.com/](https://docs.smith.langchain.com/)
*   **LangChain Hub**: Explore and pull various prompts, chains, and agents.
    *   [https://smith.langchain.com/hub](https://smith.langchain.com/hub)
*   **LangChain Tracing & Debugging Guide**: Specific documentation on how to enable and use tracing within LangChain.
    *   [https://python.langchain.com/docs/langsmith/](https://python.langchain.com/docs/langsmith/)
*   **LangSmith Best Practices**: A guide on how to effectively use LangSmith for various scenarios.
    *   [https://docs.smith.langchain.com/guides](https://docs.smith.langchain.com/guides)
